# Hands-on 5 — Change detection with annual embeddings

**Chapter 4 · Working with embeddings**

AlphaEarth gives one embedding per pixel per year, 2017 to 2025. If a place changes, its embedding moves; if it does not, the embedding stays put. Change detection becomes a distance between two vectors. In this notebook you will:

1. map the Mati wildfire of 23 July 2018 by comparing 2017 with 2019, and turn the map into a burned-area polygon with a size,
2. see what happens when the change falls inside the year you compare (2018 itself),
3. read the whole 2017–2025 stack at 20 m and build a **"when did it change?"** map,
4. follow the recovery of burned pixels year by year,
5. understand the failure modes: seasonal drift, slow change, and threshold choice.

Run time about 12 minutes (reading nine years takes most of it); ~250 MB downloaded.

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from shapely.geometry import shape
from scipy import ndimage

import eoemb

AOI = (23.90, 38.00, 24.00, 38.10)
index = eoemb.load_aef_index()
with rasterio.open(eoemb.DATA_DIR / "s2_attica_20240713_rgb.tif") as src:
    s2_rgb = np.moveaxis(src.read(), 0, -1)

## 1. Before and after: 2017 vs 2019

The fire happened in July 2018, so the *annual* 2018 embedding mixes six months of green with five months of black. The clean comparison is the year before against the year after.

For unit vectors, cosine distance = 1 − dot product. Zero means "same kind of place", values near one mean "nothing in common".

In [ ]:
t0 = time.time()
raw17, transform, crs = eoemb.aef_read_bbox(AOI, 2017, index)
raw19, _, _ = eoemb.aef_read_bbox(AOI, 2019, index)
e17, e19 = eoemb.dequantize(raw17), eoemb.dequantize(raw19)
H, W = e17.shape[1:]
extent = eoemb.lonlat_extent(transform, crs, (H, W))
print(f"two years read in {time.time() - t0:.0f} s")

dist_17_19 = 1 - np.einsum("dhw,dhw->hw", e17, e19)

fig, ax = plt.subplots(1, 2, figsize=(15, 7))
ax[0].imshow(s2_rgb, extent=extent); ax[0].set_title("Sentinel-2, July 2024 (for orientation)")
im = ax[1].imshow(dist_17_19, extent=extent, cmap="magma", vmin=0, vmax=0.6)
plt.colorbar(im, ax=ax[1], shrink=0.7, label="cosine distance"); ax[1].set_title("AlphaEarth: distance between 2017 and 2019")
plt.tight_layout()

The burned area jumps out without any tuning, and so do a few smaller changes (construction sites, cleared plots). Most of the landscape sits near zero: the embeddings of stable places are stable across years, which is not a given for a machine-learning product and is the property that makes this work.

### From a distance map to a burned-area polygon

What threshold? Look at the distribution first.

In [ ]:
vals = dist_17_19[~np.isnan(dist_17_19)]
p50, p90, p99 = np.percentile(vals, [50, 90, 99])
print(f"median {p50:.3f}   90th pct {p90:.3f}   99th pct {p99:.3f}")

plt.figure(figsize=(8, 3.5))
plt.hist(vals, bins=200, range=(0, 0.8), color="gray", log=True)
for v, lab in [(p90, "p90"), (p99, "p99"), (0.15, "0.15")]:
    plt.axvline(v, ls="--", label=f"{lab} = {v:.2f}")
plt.xlabel("cosine distance 2017 → 2019"); plt.ylabel("pixels (log)"); plt.legend();

Half of the pixels moved by less than 0.02 and 90 % by less than 0.08; the long tail is real change. A threshold of **0.15** sits clearly above the noise. As in Hands-on 3, the threshold is a decision you make by looking, and you should say so in your report. We also close small gaps (`binary_closing`): a burned settlement is a patchwork of houses, gardens and pines that did not all move equally.

In [ ]:
THR = 0.15
mask = ndimage.binary_closing(dist_17_19 > THR, iterations=3)
lab, n = ndimage.label(mask)
sizes = ndimage.sum(mask, lab, index=np.arange(1, n + 1))
big = np.isin(lab, np.flatnonzero(sizes >= 500) + 1)              # keep patches ≥ 5 ha

polys = [shape(g) for g, v in features.shapes(big.astype(np.uint8), mask=big, transform=transform)]
change = gpd.GeoDataFrame(geometry=polys, crs=crs)
change["area_ha"] = change.area / 10_000
change = change.sort_values("area_ha", ascending=False).reset_index(drop=True)
print(f"{len(change)} change polygons ≥ 5 ha; largest = {change.area_ha.iloc[0]:.0f} ha")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(s2_rgb, extent=extent)
change.to_crs(4326).boundary.plot(ax=ax, color="yellow", linewidth=1.5)
ax.set_title(f"changed 2017→2019 (distance > {THR}, patches ≥ 5 ha)");

The largest polygon (about 470 ha) and its neighbours (about 600 ha in total) trace the Mati fire inside our window. Published estimates for the whole fire range from about 1 260 ha (National Observatory of Athens) to about 1 430 ha; part of the burned area lies east of 24° E, outside our tile, and along the coast the signal is blurred by rebuilding. Try 0.12 and 0.25 yourself: the mapped area roughly halves or doubles. Not bad for one threshold on one number per pixel, but note what we did *not* get: the embedding tells us *that* the place changed, not *what* the change was. The fire interpretation is ours.

## 2. The trap of the change year

Compare 2017 with 2018 and 2018 with 2019.

In [ ]:
raw18, _, _ = eoemb.aef_read_bbox(AOI, 2018, index)
e18 = eoemb.dequantize(raw18)
d_17_18 = 1 - np.einsum("dhw,dhw->hw", e17, e18)
d_18_19 = 1 - np.einsum("dhw,dhw->hw", e18, e19)

fig, ax = plt.subplots(1, 3, figsize=(20, 6.5))
for a, d, t in zip(ax, [d_17_18, d_18_19, dist_17_19], ["2017 → 2018", "2018 → 2019", "2017 → 2019"]):
    a.imshow(d, extent=extent, cmap="magma", vmin=0, vmax=0.6); a.set_title(t)
plt.tight_layout()

fire = big & (lab == lab[big].flat[np.argmax(np.bincount(lab[big].ravel()))])  # pixels of the largest polygon
print("mean distance inside the fire perimeter:  2017→2018 = %.2f   2018→2019 = %.2f   2017→2019 = %.2f"
      % (np.nanmean(d_17_18[fire]), np.nanmean(d_18_19[fire]), np.nanmean(dist_17_19[fire])))

The change is split between the two steps, and each is weaker than the before/after comparison. An annual embedding is a *summary of the year*: an event inside the year is diluted. If you know when the event happened, compare the years around it. If you do not, the next section shows how to find out.

## 3. The full stack: when did each pixel change most?

Nine years at 10 m is 600 MB of int8; at 20 m (`factor=2`) it is 150 MB. Reading takes 5–8 minutes (the COG overview blocks still span the full tile width), so start the cell and read on. Each 20 m pixel is the mean embedding of four 10 m pixels, re-normalised.

In [ ]:
YEARS = list(range(2017, 2026))
t0 = time.time()
stack = {}
for y in YEARS:
    r, tr2, _ = eoemb.aef_read_bbox(AOI, y, index, factor=2)
    stack[y] = r
print(f"{len(YEARS)} years at 20 m in {time.time() - t0:.0f} s; each {r.shape}")
H2, W2 = r.shape[1:]
extent2 = eoemb.lonlat_extent(tr2, crs, (H2, W2))

In [ ]:
# distance between consecutive years, one map per step
steps = np.full((len(YEARS) - 1, H2, W2), np.nan, dtype=np.float32)
prev = eoemb.dequantize(stack[YEARS[0]])
for i, y in enumerate(YEARS[1:]):
    cur = eoemb.dequantize(stack[y])
    steps[i] = 1 - np.einsum("dhw,dhw->hw", prev, cur)
    prev = cur

import warnings
warnings.simplefilter("ignore", RuntimeWarning)             # all-NaN columns (nodata) are expected here
max_step = np.nanmax(steps, axis=0)                     # how big was the biggest yearly jump
when = np.nanargmax(np.nan_to_num(steps, nan=-1), axis=0)   # index of that jump
when_year = np.array(YEARS[1:])[when].astype(float)
when_year[np.isnan(max_step) | (max_step < 0.2)] = np.nan   # only show pixels that really changed

fig, ax = plt.subplots(1, 2, figsize=(15, 7))
im0 = ax[0].imshow(max_step, extent=extent2, cmap="magma", vmin=0, vmax=0.6); ax[0].set_title("largest year-to-year distance, 2017–2025")
plt.colorbar(im0, ax=ax[0], shrink=0.7)
im1 = ax[1].imshow(when_year, extent=extent2, cmap="tab10", vmin=2017.5, vmax=2027.5); ax[1].set_title("year of the largest jump (where jump > 0.2)")
cb = plt.colorbar(im1, ax=ax[1], shrink=0.7, ticks=YEARS[1:]); cb.ax.set_yticklabels([f"→{y}" for y in YEARS[1:]])
plt.tight_layout()

Two events dominate: the 2018 Mati fire near the coast (jump into 2018 and 2019, split across the two because the fire fell mid-2018) and the **August 2024 Penteli fire** on the forested slopes in the north-west (jump into 2024 and 2025, for the same reason). Notice that the Penteli jump is weaker (0.2–0.3) than Mati's (0.3–0.6): burning pine forest and shrubland moves the vector less than burning a pine-covered residential area that is then rebuilt. There is also a cluster of change into 2023 around 23.91° E, 38.03° N that we have not identified; exercise for the reader, with Sentinel-2 imagery. Scattered pixels elsewhere are construction, quarrying and land clearing. In one figure you have a change *inventory* for nine years, computed from 150 MB of data, with no training.

## 4. Recovery: following the burned pixels through time

For the pixels inside the Mati perimeter, how similar is each year to 2017, the pre-fire state?

In [ ]:
fire2 = eoemb.resample_like(fire, transform, tr2, (H2, W2), crs, method="max")   # fire mask on the 20 m grid
base = eoemb.dequantize(stack[2017])
traj = []
for y in YEARS:
    cur = eoemb.dequantize(stack[y])
    sim = np.einsum("dhw,dhw->hw", base, cur)
    traj.append({"year": y, "burned": np.nanmean(sim[fire2]), "unburned": np.nanmean(sim[~fire2 & ~np.isnan(sim)])})
traj = pd.DataFrame(traj).set_index("year")

plt.figure(figsize=(8, 4.5))
plt.plot(traj.index, traj.burned, "o-", label="pixels inside the 2018 fire perimeter")
plt.plot(traj.index, traj.unburned, "s--", color="gray", label="all other pixels")
plt.axvline(2018.55, color="red", ls=":", label="fire, July 2018")
plt.ylabel("cosine similarity to the 2017 embedding"); plt.ylim(0.5, 1.01); plt.legend(); plt.grid(alpha=.3)
plt.title("Recovery trajectory");
traj.round(3)

Unburned pixels stay at 0.92–0.97 similarity to their 2017 selves: that is the year-to-year noise floor (weather, sensor mix, model quirks), and the dip in 2024 hints at a drought year or a change in the inputs that moved *everything* a little. Burned pixels drop to about 0.67 in 2019, then climb back as vegetation regrows and houses are rebuilt, but seven years on they have not returned to their 2017 state. Whether they ever will is a question the embedding can keep answering every year, for free.

## 5. What can go wrong

- **Slow change** (urban sprawl over a decade, gradual forest degradation) produces small yearly steps that never cross a threshold. Compare distant years, or fit a trend to the trajectory instead of thresholding steps.
- **Drift without change.** A drought year, an unusual snow season or a change in the satellite constellation moves *all* embeddings a little. Judge a pixel's change against the distribution of its year, not against a fixed number. The 2017→2019 histogram above is how you find the noise floor.
- **Annual means hide within-year events**: a crop rotation, a temporary quarry, a flood that receded. If timing matters, you need per-date embeddings (Major TOM has them, Hands-on 6) or the imagery itself.
- **A distance is not a label.** Fire, harvest, construction and flooding all move the vector. To say *what* changed, classify the before and after states (Hands-on 4) or look at the images.

---
## Exercises

**E1.** Map the August 2024 Penteli fire: compare 2023 with 2025 at 10 m, threshold, polygonise, and report the burned area inside the window.

**E2.** Repeat the recovery trajectory for the 2024 fire (only one year of recovery so far) and for a built-up reference area (Nea Makri). Does the town drift from its 2017 state?

**E3.** Use TESSERA instead of AlphaEarth for 2017 vs 2019 (`eoemb.tessera_read_like`, remember to normalise). Do the two products agree on the fire perimeter? Overlay the two polygons.